In [3]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex")

library(dplyr)
library(WGCNA)
library(data.table)

source("/mnt/lareaulab/reliscu/code/FindModules/FindModules.R")

Allowing multi-threading with up to 48 threads.


Here I run FM to (hopefully) find modules representing each of the cell types present in the single-cell data

## Run 1: no batch correction

In [ ]:
data_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed"
expr <- fread("GTEx_cortex_counts_TMMF_SampleNetworks/All_02-25-12/GTEx_cortex_counts_TMMF_All_501_outliers_removed.csv", data.table=FALSE)
colnames(expr)[1] <- "Gene"

# For duplicate genes, choose row with highest mean expression

mean_expr <- data.frame(
    Index=1:nrow(expr), 
    Gene=expr[,1], 
    Expr=rowMeans(expr[,-1])
)

mean_expr <- mean_expr %>%
    group_by(Gene) %>%
    slice_max(Expr, with_ties=FALSE)

print(dim(mean_expr))

expr <- expr[mean_expr$Index,]

In [ ]:
# Subset to genes in the top X percentile for generating modules

prob <- .6
mean_expr <- rowMeans(expr[,-1])

subset_cutoff <- unname(quantile(mean_expr, prob))
print(paste("quantile(mean_expr, prob):", round(subset_cutoff, 3)))
subset <- mean_expr >= subset_cutoff
sum(subset)

In [ ]:
# Order samples by covariates of interest

sampleinfo <- read.csv("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/cortex/GTEx_cortex_sampleinfo.csv")

sampleinfo[,1] <- make.names(sampleinfo[,1])
sampleinfo <- sampleinfo[sampleinfo[,1] %in% colnames(expr),]

sampleinfo$Mean_age <- sapply(strsplit(sampleinfo$AGE, "-"), function(x) mean(as.numeric(x)))
sampleinfo$SAMPID <- make.names(sampleinfo$SAMPID)

sampleinfo <- sampleinfo %>% arrange(Mean_age)
    
expr <- expr[, c(1, match(sampleinfo[,1], colnames(expr)[-1]) + 1)]
all.equal(sampleinfo[,1], colnames(expr)[-1])

In [ ]:
samplegroups <- as.factor(sampleinfo$AGE)
merge.param <- 0.95
projectname <- paste0(data_source, "_mergeParam", merge.param, "_subsetCutoff", round(subset_cutoff, 3))

In [ ]:
FindModules(
  projectname=projectname,
  expr=expr,
  geneinfo=1,
  sampleindex=2:ncol(expr),
  samplegroups=samplegroups,
  subset=subset,
  simMat=NULL,
  saveSimMat=FALSE,
  simType="Bicor",
  beta=1,
  overlapType="None",
  TOtype="signed",
  TOdenom="min",
  MIestimator="mi.mm",
  MIdisc="equalfreq",
  signumType="rel",
  iterate=TRUE,
  signumvec=c(.99999,.9999,.999,.99,.98,.97,.96), 
  minsizevec=c(3,4,6,8,10), 
  signum=NULL,
  minSize=NULL,
  merge.by="ME",
  merge.param=merge.param,
  export.merge.comp=T,
  ZNCcut=2,
  calcSW=FALSE,
  loadTree=FALSE,
  writeKME=TRUE,
  calcBigModStat=FALSE,
  writeModSnap=TRUE
)

## Run 2: batch correction, additonal samples removed

In [ ]:
data_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_All_370_outliers_removed"
expr <- fread("GTEx_cortex_counts_TMMF_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_SampleNetworks/All_02-59-24/GTEx_cortex_counts_All_501_outliers_removed_TMM_ComBat_SMGEBTCH_corrected_All_370_outliers_removed.csv", data.table=FALSE)
colnames(expr)[1] <- "Gene"

In [ ]:
# Subset to genes in the top X percentile for generating modules

prob <- .6
mean_expr <- rowMeans(expr[,-1])

subset_cutoff <- unname(quantile(mean_expr, prob))
print(paste("quantile(mean_expr, prob):", round(subset_cutoff, 3)))
subset <- mean_expr >= subset_cutoff
sum(subset)

In [ ]:
# Order samples by covariates of interest

sampleinfo <- read.csv("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/cortex/GTEx_cortex_sampleinfo.csv")

sampleinfo[,1] <- make.names(sampleinfo[,1])
sampleinfo <- sampleinfo[sampleinfo[,1] %in% colnames(expr),]

sampleinfo$Mean_age <- sapply(strsplit(sampleinfo$AGE, "-"), function(x) mean(as.numeric(x)))
sampleinfo$SAMPID <- make.names(sampleinfo$SAMPID)

sampleinfo <- sampleinfo %>% arrange(Mean_age)
    
expr <- expr[, c(1, match(sampleinfo[,1], colnames(expr)[-1]) + 1)]
all.equal(sampleinfo[,1], colnames(expr)[-1])

In [ ]:
samplegroups <- as.factor(sampleinfo$AGE)
merge.param <- 0.95
projectname <- paste0(data_source, "_mergeParam", merge.param, "_subsetCutoff", round(subset_cutoff, 3))

In [ ]:
FindModules(
  projectname=projectname,
  expr=expr,
  geneinfo=1,
  sampleindex=2:ncol(expr),
  samplegroups=samplegroups,
  subset=subset,
  simMat=NULL,
  saveSimMat=FALSE,
  simType="Bicor",
  beta=1,
  overlapType="None",
  TOtype="signed",
  TOdenom="min",
  MIestimator="mi.mm",
  MIdisc="equalfreq",
  signumType="rel",
  iterate=TRUE,
  signumvec=c(.9999,.999,.99,.98,.97,.96,.95), 
  minsizevec=c(3,4,6,8,10), 
  signum=NULL,
  minSize=NULL,
  merge.by="ME",
  merge.param=merge.param,
  export.merge.comp=T,
  ZNCcut=2,
  calcSW=FALSE,
  loadTree=FALSE,
  writeKME=TRUE,
  calcBigModStat=FALSE,
  writeModSnap=TRUE
)